# Deep Hedging — Phase 1, brique 3 : le delta-hedging discret

On assemble les briques 1 et 2. On **vend un call**, donc on devra payer son payoff en T, et on se couvre en détenant `delta` unités du sous-jacent, rééquilibrées à chaque date. On mesure l'**erreur de couverture** : le P&L final de cette position couverte.

Deux résultats à observer, tous les deux sans coûts de transaction :
1. la moyenne du P&L est ~0 (la prime finance la réplication),
2. l'écart-type décroît comme `1/sqrt(n_steps)` quand on rééquilibre plus souvent : la théorie du temps continu se confirme.

Ce delta-hedging est le **benchmark** que le réseau de la phase 3 devra battre une fois qu'on aura ajouté les coûts.

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
RNG = np.random.default_rng(0)

## Les briques précédentes, rassemblées (notebook autonome)

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, rng=RNG):
    dt = T/n_steps
    Z = rng.standard_normal((n_paths, n_steps))
    inc = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    logp = np.concatenate([np.zeros((n_paths,1)), np.cumsum(inc, axis=1)], axis=1)
    return S0*np.exp(logp)

def bs_price(S, K, tau, r, sigma):
    S = np.asarray(S, float)
    d1 = (np.log(S/K) + (r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    return S*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)

def bs_delta(S, K, tau, r, sigma):
    S = np.asarray(S, float)
    d1 = (np.log(S/K) + (r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau))
    return norm.cdf(d1)

## Le simulateur de couverture

On suit un **compte de cash** auto-financé, trajectoire par trajectoire (tout est vectorisé sur les trajectoires) :

- en t_0, on encaisse la prime `C_0` (on vend le call),
- à chaque date on rééquilibre vers `delta_k`, en payant les actions achetées (et les coûts, nuls pour l'instant),
- entre deux dates, le cash capitalise au taux `r`,
- en T, on liquide les actions et on paie le payoff `(S_T - K)+`.

Le P&L final est `cash + actions·S_T - payoff`. Pour une couverture parfaite continue, il vaudrait exactement 0.

In [ ]:
def delta_hedge_pnl(S, K, T, r, sigma, cost=0.0):
    """P&L final d'un vendeur de call qui se delta-couvre. S : (n_paths, n_steps+1)."""
    n_paths, n1 = S.shape
    n_steps = n1 - 1
    dt = T / n_steps
    times = np.linspace(0, T, n1)

    cash = bs_price(S[:, 0], K, T, r, sigma).copy()   # prime encaissée en vendant le call
    shares = np.zeros(n_paths)                         # aucune action au départ

    for k in range(n_steps):
        tau = T - times[k]                             # temps restant à cette date
        delta_k = bs_delta(S[:, k], K, tau, r, sigma)  # delta cible, par trajectoire
        trade = delta_k - shares                       # actions à acheter (ou vendre)
        cash -= trade * S[:, k]                        # on paie ces actions
        cash -= cost * np.abs(trade) * S[:, k]         # coût de transaction (0 ici)
        shares = delta_k                               # nouvelle position
        cash *= np.exp(r * dt)                          # le cash capitalise jusqu'à la date suivante

    payoff = np.maximum(S[:, -1] - K, 0.0)             # ce qu'on doit payer en T
    return cash + shares * S[:, -1] - payoff           # P&L final

## Vérification : moyenne ~0 et écart-type en 1/sqrt(n)

In [ ]:
S0, K, mu, r, sigma, T = 100., 100., 0.10, 0.02, 0.20, 1.0
n_paths = 100_000
ns = [5, 21, 63, 252, 1000]

print(f"{'n_steps':>8} | {'moyenne P&L':>12} | {'std P&L':>10} | {'std*sqrt(n)':>12}")
stds = []
for n in ns:
    S = simulate_gbm(S0, mu, sigma, T, n, n_paths)
    pnl = delta_hedge_pnl(S, K, T, r, sigma, cost=0.0)
    stds.append(pnl.std())
    print(f"{n:>8} | {pnl.mean():>12.4f} | {pnl.std():>10.4f} | {pnl.std()*np.sqrt(n):>12.3f}")

## Visualisation

À gauche, la distribution du P&L pour deux fréquences : plus on rééquilibre, plus elle se resserre autour de 0. À droite, l'écart-type en fonction de `n_steps` en échelle log-log : une droite de pente -1/2, la signature du `1/sqrt(n)`.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for n, c in [(21, "crimson"), (252, "navy")]:
    S = simulate_gbm(S0, mu, sigma, T, n, n_paths)
    pnl = delta_hedge_pnl(S, K, T, r, sigma)
    a1.hist(pnl, bins=120, density=True, alpha=0.5, label=f"n_steps={n} (std={pnl.std():.3f})")
a1.set_title("Erreur de couverture (sans coûts)"); a1.set_xlabel("P&L final"); a1.legend()

a2.loglog(ns, stds, "o-")
a2.set_title("Écart-type de l'erreur vs fréquence")
a2.set_xlabel("n_steps"); a2.set_ylabel("std du P&L"); a2.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()